# MSGraphMCP Diagnostics KQL Notebook
Use these KQL queries in the Log Analytics workspace linked to Application Insights.

This notebook focuses on: MCP tool calls, latency spikes, dependency failures, correlation traces, and origin outage signals (including 504 scenarios).

In [ ]:
// 1) MCP tool latency and failure summary
AppRequests
| where TimeGenerated > ago(24h)
| extend ToolName = tostring(Properties["mcp.tool_name"])
| where isnotempty(ToolName)
| summarize calls=count(), failures=countif(Success == false or toint(ResultCode) >= 400), p50=percentile(DurationMs, 50), p95=percentile(DurationMs, 95), p99=percentile(DurationMs, 99) by ToolName
| extend errorRatePct = round(100.0 * todouble(failures) / todouble(calls), 2)
| order by failures desc, p95 desc

In [ ]:
// 2) MCP requests and errors by 5-minute bin
AppRequests
| where TimeGenerated > ago(24h)
| where Name has "/mcp" or tostring(Properties["mcp.method"]) in ("initialize", "tools/call")
| summarize calls=count(), failures=countif(Success == false or toint(ResultCode) >= 400), p95=percentile(DurationMs, 95) by bin(TimeGenerated, 5m)
| extend errorRatePct = iff(calls == 0, 0.0, round(100.0 * todouble(failures) / todouble(calls), 2))
| order by TimeGenerated desc

In [ ]:
// 3) Dependency failures (Graph and outbound HTTP)
AppDependencies
| where TimeGenerated > ago(24h)
| where Success == false or toint(ResultCode) >= 400
| project TimeGenerated, DependencyType, Name, Target, ResultCode, DurationMs, OperationId, OperationName
| order by TimeGenerated desc

In [ ]:
// 4) Correlated waterfall for one transport session
let targetTransportSession = "<mcp-transport-session-id>";
union
(
    AppRequests
    | extend ItemType = "request", Message = Name, Duration = DurationMs, SuccessFlag = tostring(Success)
    | extend TransportSessionId = tostring(Properties["mcp.transport_session_id"])
    | where TransportSessionId == targetTransportSession
    | project TimeGenerated, ItemType, Message, Duration, ResultCode, SuccessFlag, OperationId, OperationName
),
(
    AppDependencies
    | extend ItemType = "dependency", Message = strcat(DependencyType, ": ", Name, " -> ", Target), Duration = DurationMs, SuccessFlag = tostring(Success)
    | where OperationId in (
        AppRequests
        | extend TransportSessionId = tostring(Properties["mcp.transport_session_id"])
        | where TransportSessionId == targetTransportSession
        | project OperationId
    )
    | project TimeGenerated, ItemType, Message, Duration, ResultCode, SuccessFlag, OperationId, OperationName
),
(
    AppTraces
    | extend ItemType = "trace", Duration = real(null), ResultCode = "", SuccessFlag = ""
    | where OperationId in (
        AppRequests
        | extend TransportSessionId = tostring(Properties["mcp.transport_session_id"])
        | where TransportSessionId == targetTransportSession
        | project OperationId
    )
    | project TimeGenerated, ItemType, Message, Duration, ResultCode, SuccessFlag, OperationId, OperationName
)
| order by TimeGenerated asc

In [ ]:
// 5) Request-gap detector (possible origin downtime)
AppRequests
| where TimeGenerated > ago(24h)
| summarize calls=count() by bin(TimeGenerated, 5m)
| order by TimeGenerated asc
| extend likelyGap = iff(calls == 0, 1, 0)

In [ ]:
// 6) ACI stop/start/restart operations from Azure Activity (requires ingestion)
AzureActivity
| where TimeGenerated > ago(7d)
| where ResourceProviderValue =~ "MICROSOFT.CONTAINERINSTANCE"
| where Resource =~ "msgraph-mcp-27992"
| where OperationNameValue has_any ("/stop/action", "/start/action", "/restart/action")
| project TimeGenerated, OperationNameValue, ActivityStatusValue, Caller, ResourceGroup, ResourceId
| order by TimeGenerated desc

In [ ]:
// 7) Front Door 504 OriginTimeout evidence (requires Front Door diagnostics to LA)
AzureDiagnostics
| where TimeGenerated > ago(24h)
| where ResourceProvider == "MICROSOFT.CDN"
| where Category has "FrontDoor"
| where toint(httpStatusCode_s) == 504 or errorInfo_s has "OriginTimeout"
| project TimeGenerated, host_s, requestUri_s, httpStatusCode_s, errorInfo_s, clientIp_s
| order by TimeGenerated desc